In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import sys

# point this to the hydroviz repo to get the luts
sys.path.insert(0, "/beegfs/CMIP6/jdpaul3/hydroviz/data/preprocess")
from luts import *

In [2]:
# outputs from .../hydroviz/data/preprocess/run_build_nc.py
seg = xr.open_dataset("/beegfs/CMIP6/jdpaul3/hydroviz_data/maurer/nc_stats_fix/seg.nc")
seg_diff = xr.open_dataset("/beegfs/CMIP6/jdpaul3/hydroviz_data/maurer/nc_stats_fix/seg_diff.nc")

In [3]:
# Sanitize seg_diff: replace -999999 sentinel fill values and inf with NaN before applying change signals.
# Both can enter via upstream ratio computation (0-division) or missing-event fill values.
# This assures we have NaNs downstream instead of large negative values that can skew change signal computations.
FILL_VALUE = -999999.0
seg_diff = seg_diff.where((seg_diff != FILL_VALUE) & np.isfinite(seg_diff))

In [4]:
# use the reverse_encodings_lookup dict to convert the encoded dimension values back to their original string values
# this helps us be more explicit for the workflow below, and also makes it easier to debug if we need to inspect the data
for dim in ["landcover", "model", "scenario", "era"]:
    seg = seg.assign_coords({dim: [reverse_encodings_lookup[dim][int(v)] for v in seg[dim].values]})
    seg_diff = seg_diff.assign_coords({dim: [reverse_encodings_lookup[dim][int(v)] for v in seg_diff[dim].values]})

In [5]:
JULIAN_DATE_VARS = {"spr_ord", "sum_ord", "th1", "tl1"}
DAYS_PER_YEAR_VARS = {"dh15", "dl16", "lf1", "ra8"}

def build_seg_corrected(seg, seg_diff):
    # function to apply the GCM change signal (from seg_diff) to the Maurer historical baseline (from seg) to produce a corrected dataset

    def _apply_change_signal(var, base, delta):
        if delta.attrs["difference_method"] == "ratio":
            result = base * delta
        elif var in JULIAN_DATE_VARS:
            # Modulo wrapping keeps Julian dates within [1, 366].
            # Subtract 1 before and add 1 after to prevent day 0 as a valid output.
            result = (base + delta - 1) % 366 + 1
        else:
            result = base + delta
        # Number-of-days-per-year variables cannot physically exceed 366.
        if var in DAYS_PER_YEAR_VARS:
            result = result.clip(max=366)
        return result

    # Maurer historical 1976-2005 is the baseline applied to all GCM change signals
    maurer_hist = seg.sel(model="Maurer", scenario="historical", era="1976-2005")

    # For each variable, apply the GCM change signal (from seg_diff) to the Maurer baseline.
    # xarray broadcasting propagates maurer_hist (landcover, stream_id) across the
    # (landcover, model, scenario, era, stream_id) dims of seg_diff automatically.
    corrected_gcm = xr.Dataset(
        {var: _apply_change_signal(var, maurer_hist[var], seg_diff[var]) for var in seg.data_vars},
        attrs=seg.attrs,
    )

    # Expand to the full coordinate space of seg; slots with no correction become NaN
    seg_corrected = corrected_gcm.reindex_like(seg, fill_value=np.nan)

    # Re-insert the Maurer historical baseline into its own slot
    maurer_expanded = maurer_hist.expand_dims(
        {"model": ["Maurer"], "scenario": ["historical"], "era": ["1976-2005"]}
    )
    seg_corrected = seg_corrected.combine_first(maurer_expanded)

    return seg_corrected

In [6]:
seg_corrected = build_seg_corrected(seg, seg_diff)

In [7]:
# reindex seg_diff to have the same coordinate space as seg, filling in missing slots with NaN
seg_diff_reindexed = seg_diff.reindex_like(seg, fill_value=np.nan)

# combine the original seg, the reindexed seg_diff, and the seg_corrected into one dataset with a new "source" dimension to indicate where each value came from
seg_combined = xr.concat(
    [seg, seg_diff_reindexed, seg_corrected],
    dim=pd.Index(["original_gcm", "gcm_diff", "gcm_diff_applied_to_maurer"], name="source")
)

In [8]:
# Add difference_method attribute to each variable from seg_diff attrs
for var in seg_combined.data_vars:
    seg_combined[var].attrs["difference_method"] = seg_diff[var].attrs["difference_method"]

# Add ma99: mean of monthly flow variables, use NaN for gcm_diff source
monthly_vars = ["ma12", "ma13", "ma14", "ma15", "ma16", "ma17", "ma18", "ma19", "ma20", "ma21", "ma22", "ma23"]
ma99 = xr.concat([seg_combined[v] for v in monthly_vars], dim="month").mean("month")
ma99 = ma99.where(ma99.source != "gcm_diff")
ma99.attrs["difference_method"] = ""
ma99.attrs["description"] = "Mean annual flow, calculated as the mean of monthly flow variables (ma12-ma23)"
ma99.attrs["units"] = seg_combined["ma12"].attrs["units"]
seg_combined = seg_combined.assign(ma99=ma99)

In [9]:
seg_combined

<xarray.Dataset> Size: 20GB
Dimensions:    (source: 3, landcover: 2, model: 14, scenario: 5, era: 4,
                stream_id: 56460)
Coordinates:
  * stream_id  (stream_id) int32 226kB 1 2 3 4 5 ... 56457 56458 56459 56460
  * landcover  (landcover) <U7 56B 'dynamic' 'static'
  * model      (model) object 112B 'ACCESS1-0' 'BCC-CSM1-1' ... 'NorESM1-M'
  * scenario   (scenario) <U10 200B 'historical' 'rcp26' 'rcp45' 'rcp60' 'rcp85'
  * era        (era) <U9 144B '1976-2005' '2016-2045' '2046-2075' '2071-2100'
  * source     (source) object 24B 'original_gcm' ... 'gcm_diff_applied_to_ma...
Data variables: (12/53)
    dh1        (source, landcover, model, scenario, era, stream_id) float32 379MB ...
    dh2        (source, landcover, model, scenario, era, stream_id) float32 379MB ...
    dh3        (source, landcover, model, scenario, era, stream_id) float32 379MB ...
    dh4        (source, landcover, model, scenario, era, stream_id) float32 379MB ...
    dh5        (source, landcover, model, scenario, era, stream_id) float32 379MB ...
    dh15       (source, landcover, model, scenario, era, stream_id) float32 379MB ...
    ...         ...
    ra8        (source, landcover, model, scenario, era, stream_id) float32 379MB ...
    spr_ord    (source, landcover, model, scenario, era, stream_id) float32 379MB ...
    sum_ord    (source, landcover, model, scenario, era, stream_id) float32 379MB ...
    th1        (source, landcover, model, scenario, era, stream_id) float32 379MB ...
    tl1        (source, landcover, model, scenario, era, stream_id) float32 379MB ...
    ma99       (source, landcover, model, scenario, era, stream_id) float32 379MB ...
Attributes:
    Data Source:         {'Title': 'Model Input and Output for Hydrologic Sim...
    CMIP5 GCM Metadata:  {'ACCESS1-0': {'Modeling Center': 'Commonwealth Scie...

In [10]:
# Encode dimension coordinates back to integers (float32) and populate encoding attrs.
# this gets the dataset ready for Rasdaman ingest

source_encoding = {0: "original_gcm", 1: "gcm_diff", 2: "gcm_diff_applied_to_maurer"}

for dim in ["landcover", "model", "scenario", "era"]:
    encoding_dict = reverse_encodings_lookup[dim]  # int -> string
    str_to_int = {v: k for k, v in encoding_dict.items()}
    encoded_values = np.array([str_to_int[v] for v in seg_combined[dim].values], dtype="float32")
    seg_combined = seg_combined.assign_coords({dim: encoded_values})
    seg_combined[dim].attrs["encoding"] = str(encoding_dict)

source_str_to_int = {v: k for k, v in source_encoding.items()}
encoded_source = np.array([source_str_to_int[v] for v in seg_combined["source"].values], dtype="float32")
seg_combined = seg_combined.assign_coords({"source": encoded_source})
seg_combined["source"].attrs["encoding"] = str(source_encoding)

# Sort all encoded dimensions so coordinate values are monotonically increasing.
# combine_first in build_seg_corrected can lexicographically reorder string coords
# (e.g. 'Maurer' sorts after 'MRI-CGCM3' in ASCII), which scrambles the integer
# encoding positions and causes Rasdaman to map models to wrong indices.
seg_combined = seg_combined.sortby(["source", "landcover", "model", "scenario", "era"])

seg_combined.to_netcdf("/beegfs/CMIP6/jdpaul3/hydroviz_data/maurer/nc_stats_fix/seg_combined.nc")